# GeoLocate End-to-End Pipeline (SageMaker Ready)

This notebook is designed to run on a completely new SageMaker machine.

Pipeline stages:
1. Bootstrap or clone repository
2. Install dependencies
3. Configure Kaggle credentials
4. Download dataset cache
5. Prepare manifest and splits
6. Optional smoke test
7. Train model
8. Evaluate model and render confusion matrix

Before running all cells:
- If this notebook is not inside your cloned repository, set `REPO_URL` in Cell 3.
- Provide Kaggle access using SageMaker environment variables `KAGGLE_USERNAME` and `KAGGLE_KEY`, or a `~/.kaggle/kaggle.json` file.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

START_DIR = Path.cwd()
HOME_DIR = Path.home()
print(f"Start directory: {START_DIR}")
print(f"Home directory: {HOME_DIR}")

def run_step(cmd, *, cwd=None, env=None):
    """Run a command and stream output to the notebook in real time."""
    if isinstance(cmd, str):
        raise TypeError("cmd must be a list of arguments, not a shell string")

    print(f"\n$ {' '.join(cmd)}")
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd or Path.cwd()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")

    ret = process.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed with exit code {ret}: {' '.join(cmd)}")

def ensure_command_exists(command):
    """Fail fast if a required binary is missing."""
    from shutil import which
    if which(command) is None:
        raise RuntimeError(f"Required command not found on PATH: {command}")

PYTHON = sys.executable
print(f"Python executable: {PYTHON}")

In [ ]:
# SageMaker bootstrap: use current repo if present, otherwise clone.
REPO_URL = ""  # Example: "https://github.com/<owner>/GeoLocate.git"
REPO_BRANCH = "main"
REPO_PARENT_DIR = HOME_DIR / "SageMaker"
REPO_DIR_NAME = "GeoLocate"
EXPECTED_FILES = ["train.py", "prepare_dataset.py", "download_dataset.py", "evaluate.py", "requirements.txt"]

def looks_like_repo(path):
    return all((path / name).exists() for name in EXPECTED_FILES)

if looks_like_repo(START_DIR):
    PROJECT_ROOT = START_DIR
    print(f"Using existing repository at: {PROJECT_ROOT}")
else:
    ensure_command_exists("git")
    REPO_PARENT_DIR.mkdir(parents=True, exist_ok=True)
    PROJECT_ROOT = REPO_PARENT_DIR / REPO_DIR_NAME

    if not PROJECT_ROOT.exists():
        if not REPO_URL.strip():
            raise RuntimeError(
                "Repository not found in current directory. Set REPO_URL in this cell and re-run."
            )
        run_step([
            "git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(PROJECT_ROOT)
        ])
    else:
        print(f"Repository directory already exists: {PROJECT_ROOT}")

    if not looks_like_repo(PROJECT_ROOT):
        raise RuntimeError(
            f"Directory exists but does not look like GeoLocate repo: {PROJECT_ROOT}"
        )

os.chdir(PROJECT_ROOT)
PYTHON = sys.executable
print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory now: {Path.cwd()}")
print(f"Python executable: {PYTHON}")

In [ ]:
# Execution toggles for first-run vs repeat-run workflows.
RUN_INSTALL_DEPS = True
RUN_DOWNLOAD = True
RUN_PREPARE = True
RUN_SMOKE_TEST = False
RUN_TRAIN = True
RUN_EVALUATE = True

print("Toggles:")
print(f"  RUN_INSTALL_DEPS={RUN_INSTALL_DEPS}")
print(f"  RUN_DOWNLOAD={RUN_DOWNLOAD}")
print(f"  RUN_PREPARE={RUN_PREPARE}")
print(f"  RUN_SMOKE_TEST={RUN_SMOKE_TEST}")
print(f"  RUN_TRAIN={RUN_TRAIN}")
print(f"  RUN_EVALUATE={RUN_EVALUATE}")

In [ ]:
if RUN_INSTALL_DEPS:
    run_step([PYTHON, "-m", "pip", "install", "--upgrade", "pip"])
    run_step([PYTHON, "-m", "pip", "install", "-r", "requirements.txt"])
else:
    print("Skipping dependency install (RUN_INSTALL_DEPS=False).")

In [ ]:
# Configure Kaggle credentials for new machines before dataset download.
if RUN_DOWNLOAD:
    kaggle_dir = HOME_DIR / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"

    if kaggle_json.exists():
        print(f"Using existing Kaggle credentials at: {kaggle_json}")
    else:
        kaggle_username = os.environ.get("KAGGLE_USERNAME", "").strip()
        kaggle_key = os.environ.get("KAGGLE_KEY", "").strip()

        if kaggle_username and kaggle_key:
            kaggle_dir.mkdir(parents=True, exist_ok=True)
            payload = "{\"username\": \"" + kaggle_username + "\", \"key\": \"" + kaggle_key + "\"}\n"
            kaggle_json.write_text(payload, encoding="utf-8")
            os.chmod(kaggle_json, 0o600)
            print(f"Wrote Kaggle credentials to: {kaggle_json}")
        else:
            raise RuntimeError(
                "Kaggle credentials are missing. Set environment variables KAGGLE_USERNAME and KAGGLE_KEY, "
                "or upload ~/.kaggle/kaggle.json, then re-run this cell."
            )
else:
    print("Skipping Kaggle credential setup because RUN_DOWNLOAD=False.")

In [ ]:
if RUN_DOWNLOAD:
    run_step([PYTHON, "download_dataset.py"], cwd=PROJECT_ROOT)
else:
    print("Skipping dataset download step.")

In [ ]:
if RUN_PREPARE:
    run_step([PYTHON, "prepare_dataset.py"], cwd=PROJECT_ROOT)
else:
    print("Skipping dataset preparation step.")

In [ ]:
manifest_path = PROJECT_ROOT / "data" / "manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest missing at {manifest_path}. Run preparation step first.")

manifest = pd.read_csv(manifest_path)
print(f"Manifest rows: {len(manifest):,}")
print("Split counts:")
print(manifest["split"].value_counts())
print("Unique sectors:", manifest["sector"].nunique())
display(manifest.head())

In [ ]:
if RUN_SMOKE_TEST:
    run_step([PYTHON, "smoke_test.py"], cwd=PROJECT_ROOT)
else:
    print("Skipping smoke test.")

In [ ]:
if RUN_TRAIN:
    run_step([PYTHON, "train.py"], cwd=PROJECT_ROOT)
else:
    print("Skipping training step.")

In [ ]:
if RUN_EVALUATE:
    run_step([PYTHON, "evaluate.py"], cwd=PROJECT_ROOT)
else:
    print("Skipping evaluation step.")

cm_path = PROJECT_ROOT / "checkpoints" / "confusion_matrix.png"
if cm_path.exists():
    print(f"Displaying {cm_path}")
    display(Image(filename=str(cm_path)))
else:
    print(f"Confusion matrix not found at {cm_path}. Run evaluation first.")